# 02 ? Protocolo comum de avalia??o

Este notebook define a bateria fixa que ser? usada pelos tr?s experimentos de forecasting e pelas variantes com equation head. N?o treina modelos: fixa testes, split, m?tricas, checkpoint selection e or?amento pareado.

## Registro experimental

- Or?amento pareado pr?-registrado: 6.000 optimizer updates por seed ? 3 seeds; 8 exemplos ? 6 canais (48 linhas por forward), contexto 512, horizonte 64. Chronos-2 default learning rate `1e-6`. Igualar updates n?o garante FLOPs iguais no curriculum, que deve reportar canais/s?rie e tempo total.

- Chronos-2: `amazon/chronos-2` @ `29ec3766d36d6f73f0696f85560a422f50e8498c`; contexto m?ximo 8192, patch de sa?da 16, m?ximo 1024 passos previstos.
- Estado: protocolo definido; os n?meros abaixo s?o quotas planejadas e n?o resultados observados.

- Status: protocolo pr?-registrado para piloto; congelar ap?s execu??o do piloto e antes de comparar resultados completos.
- Checkpoint inicial: mesmo modelo Chronos-2 e mesma revis?o em cada bra?o.
- Grupos: 8 exemplos por forward, 6 canais cada (48 s?ries/canais de entrada); usar a unidade real de `batch_size` do trainer Chronos-2.
- Contexto/horizonte de refer?ncia: 512/64, sujeitos a confirma??o de limites do checkpoint.
- Seeds de treino: 20261113, 20261114, 20261115. Seed independente de valida??o/teste abaixo.
- Nunca usar o split de teste para escolha de checkpoint, scheduler ou pesos de loss.


## Reproduction instructions

Use Python 3.11 from the repository root. Install the project dependencies with `python -m pip install -e ".[learning,dev,release]"`, then execute this notebook with:

`python -m nbconvert --to notebook --execute --inplace experiments/generator_model_evaluation/02_common_evaluation_protocol.ipynb`

The Chronos-2 checkpoint used by this experiment is pinned to `amazon/chronos-2@29ec3766d36d6f73f0696f85560a422f50e8498c`; the skeleton teacher, where used, is pinned to `mateuswiteck/tsfmxai-canonicalization-encoder-decoder` variant `transformer_skeleton` at `2fcdcc88b41125bcfd92953c0bb8a668a603d167`. The notebook records its own seeds, settings, observed result, and current limitations.


In [1]:
from pathlib import Path
import sys
ROOT=Path.cwd()
while not (ROOT/'src/tsfmxai').exists() and ROOT!=ROOT.parent: ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/'src'))
from dataclasses import dataclass
import numpy as np
import pandas as pd

MODEL_ID = 'amazon/chronos-2'
MODEL_REVISION = '29ec3766d36d6f73f0696f85560a422f50e8498c'
TRAIN_UPDATES = 6_000
LEARNING_RATE = 1e-6  # Chronos-2 pipeline default
TRAIN_SEEDS = (20261113, 20261114, 20261115)
EVAL_SEED = 20261116
CHANNELS_PER_EXAMPLE = 6
EXAMPLES_PER_FORWARD = 8
CONTEXT_LENGTH = 512
PREDICTION_LENGTH = 64
QUANTILES = (0.1, 0.5, 0.9)
EVAL_EXAMPLES_PER_GENERATOR = 512
EVAL_SYSTEMS_PER_GENERATOR = 128

@dataclass(frozen=True)
class EvaluationSplitSpec:
    generator: str
    split: str
    seed: int
    systems: int
    examples_per_system: int
    channels: int
    context_length: int
    prediction_length: int

# Freeze these specifications and persist the generated arrays under ignored data/.
SPLITS = [
    EvaluationSplitSpec(g, split, EVAL_SEED + offset, EVAL_SYSTEMS_PER_GENERATOR,
                        4, CHANNELS_PER_EXAMPLE, CONTEXT_LENGTH, PREDICTION_LENGTH)
    for g in ("lmc_synth", "tsfmxai")
    for split, offset in (("validation", 1), ("test", 2))
]
assert len({(s.generator,s.split) for s in SPLITS}) == 4
assert TRAIN_UPDATES == 6_000 and EXAMPLES_PER_FORWARD * CHANNELS_PER_EXAMPLE == 48
pd.DataFrame([s.__dict__ for s in SPLITS])

,generator,split,seed,systems,examples_per_system,channels,context_length,prediction_length
0,lmc_synth,validation,20261117,128,4,6,512,64
1,lmc_synth,test,20261118,128,4,6,512,64
2,tsfmxai,validation,20261117,128,4,6,512,64
3,tsfmxai,test,20261118,128,4,6,512,64


## Compara??es e interpreta??o

A bateria cont?m valida??o e teste de ambos os geradores: LMC-Synth e tsfmxai. Reportar quatro c?lulas (treino LMC/teste LMC; treino LMC/teste tsfmxai; treino tsfmxai/teste LMC; treino tsfmxai/teste tsfmxai), com linhas por seed. Isso mostra desempenho dentro e fora da distribui??o de treino. Acrescentar datasets reais fixos somente depois de registrar nome, vers?o, split temporal e pr?-processamento antes de executar compara??es.

O or?amento pareado para comparar os dois geradores sem equation head fixa checkpoint inicial, n?mero de canais, exemplos por forward, contexto, horizonte, n?mero de forwards e updates, acumula??o, otimizador, scheduler e precis?o. Reportar separadamente tempo de gera??o e treinamento. A variante com heads auxiliares mant?m exposi??o e updates e informa o custo adicional de computa??o.

Todos os sistemas e trajet?rias de um mecanismo ficam no mesmo split. Para o gerador tsfmxai, tamb?m reportar generaliza??o para skeletons can?nicos n?o vistos. N?o comparar janelas vizinhas de uma mesma trajet?ria entre treino e avalia??o.


In [2]:
from tsfmxai.evaluation.forecasting import point_forecast_metrics, mase, quantile_metrics

# Metric contract example: leading dimensions are [example, channel], last is horizon.
rng = np.random.default_rng(EVAL_SEED)
target = rng.normal(size=(3, 6, PREDICTION_LENGTH))
context = rng.normal(size=(3, 6, CONTEXT_LENGTH))
median = target + 0.1
quantile_predictions = np.stack([median - 0.5, median, median + 0.5])
metrics = point_forecast_metrics(median, target)
metrics['mase'] = mase(median, target, context)
metrics.update(quantile_metrics(quantile_predictions, target, QUANTILES))
metrics

{'mae': 0.10000000000000002,
 'rmse': 0.10000000000000003,
 'mase': 0.08944061205375767,
 'mean_pinball': 0.05,
 'coverage_q0.1': 0.0,
 'pinball_q0.1': 0.04,
 'coverage_q0.5': 1.0,
 'pinball_q0.5': 0.05000000000000001,
 'coverage_q0.9': 1.0,
 'pinball_q0.9': 0.060000000000000005}

## M?tricas e sele??o de checkpoint

- Forecast pontual: MAE, RMSE e MASE. MASE usa somente a escala na?ve de diferen?as do contexto, n?o estat?sticas do futuro.
- Previs?o probabil?stica: pinball loss por quantil e m?dia; cobertura emp?rica por quantil e por horizonte. Reportar tamb?m cobertura/ largura do intervalo central de 80%.
- Agrega??o: primeiro calcular por exemplo multivariado; reportar m?dia e mediana por dataset, horizonte e seed. N?o permitir que canais de uma mesma tarefa contem como exemplos independentes na incerteza estat?stica.
- Incerteza: bootstrap pareado por sistema/mecanismo (n?o por janela) para intervalo de confian?a de diferen?as entre bra?os.
- Sele??o: menor loss de valida??o, com regra e frequ?ncia id?nticas; teste uma ?nica vez ap?s sele??o.
- Equation head: similaridade/erro de embedding separado de forecasting; desvio padr?o ? inova??o em unidade original, avaliado por MAE em log-escala e correla??o; avaliar skeleton apenas com m?trica estrutural independente.
- Dados inv?lidos: registrar NaN/inf e regra de falha; n?o remover exemplos sem relatar contagens.


## Itens a congelar antes do treinamento longo

1. Checkpoint e revis?o exata do Chronos-2; janela suportada e conven??o de normaliza??o.
2. Par?metros do LMC-Synth (incluindo distribui??es Weibull/Dirichlet e banco de kernels).
3. Configura??o de gera??o tsfmxai ap?s escolha cient?fica de opera??es/lags e pol?tica de estabilidade/rejei??o.
4. Quantidade de batches/updates e regra de passagem por ?poca.
5. Revis?o e configura??o do professor Hugging Face que prev? somente skeleton (sem coeficientes), para notebooks posteriores.
6. Hardware, bibliotecas e comandos de reprodu??o.

Os n?meros de avalia??o neste rascunho s?o escala inicial para piloto, n?o quotas cient?ficas validadas.